# Stage 02 · Step 0 — Generate multi-τ training corpus

The supervised RUL head benefits from telemetry observed under several maintenance schedules, not just the single baseline τ already in `data/fleet_baseline.parquet`. This notebook draws K τ vectors via Latin-Hypercube sampling over the same ranges Stage 01 explores and runs the SDG simulator on the **train printers (id 0..69)** for each.

Outputs:
- `data/policy_runs/policy_{k:03d}.parquet` — one parquet per τ schedule (train printers only, RUL labels included).
- `data/policy_runs/manifest.json` — index of τ values per file.

Skip this notebook if the corpus already exists; the SSL pretraining notebook will fall back to `fleet_baseline.parquet` if no policy runs are found.

In [1]:
from __future__ import annotations
import json
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq
from scipy.stats import qmc

from ml_models.lib.data import TRAIN_PRINTERS
from ml_models.lib.env_runner import default_dates, run_with_tau
from ml_models import PROJECT_ROOT
from sdg.generate import load_configs
from sdg.labels import add_rul_labels
from sdg.schema import COMPONENT_IDS, table_from_dataframe

POLICY_DIR = PROJECT_ROOT / 'data/policy_runs'
POLICY_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = POLICY_DIR / 'manifest.json'

from ml_models.lib.fast import N_LHS_SCHEDULES, banner
banner()


[fast] mode=FAST · parallel=12 · trials=60/200 · K=20 · epochs=8/2 · ppo_ts=1000/8000 · seeds=(0, 1)


In [2]:
TAU_RANGES = {
    'C1': (50.0, 2_000.0),
    'C2': (500.0, 20_000.0),
    'C3': (24.0, 500.0),
    'C4': (100.0, 2_000.0),
    'C5': (500.0, 8_000.0),
    'C6': (1_000.0, 20_000.0),
}
# K = number of LHS-sampled τ schedules to simulate. The SSL pretraining corpus
# grows linearly with K (each run = ~21 MB on disk, ~70 printers × 10 years of
# rows). 5 = smoke run; 30 = surrogate-quality default; 60+ = recommended for
# Stage 03 RL+SSL because the policy will see a much richer τ distribution
# at SSL-pretrain time → better embeddings → better RL finetuning.
# Wall-clock: ~2–5 min per run on CPU, so K=60 ≈ 2–5 h. Cells below are
# resumable — re-running skips runs already on disk.
K = N_LHS_SCHEDULES  # was 60; toggled by FAST_MODE in ml_models.lib.fast
SEED = 17

sampler = qmc.LatinHypercube(d=len(TAU_RANGES), seed=SEED)
unit = sampler.random(K)
tau_schedules: list[dict[str, float]] = []
for row in unit:
    schedule = {}
    for i, (component_id, (low, high)) in enumerate(TAU_RANGES.items()):
        schedule[component_id] = float(np.exp(np.log(low) + row[i] * (np.log(high) - np.log(low))))
    tau_schedules.append(schedule)
tau_schedules

[{'C1': 155.59583785006157,
  'C2': 1765.2107916409634,
  'C3': 86.472130536287,
  'C4': 1039.6324922070855,
  'C5': 3882.5691163902784,
  'C6': 13990.349494924487},
 {'C1': 611.1034528608234,
  'C2': 537.1586666615513,
  'C3': 154.47160942624942,
  'C4': 943.5781632106193,
  'C5': 636.9234158299065,
  'C6': 1931.776617446712},
 {'C1': 1132.435398876648,
  'C2': 3804.364631233711,
  'C3': 378.572339510078,
  'C4': 1712.3245117575705,
  'C5': 1401.0340681294374,
  'C6': 5507.817068918813},
 {'C1': 1381.2400315847656,
  'C2': 2544.203495476921,
  'C3': 42.96238310854736,
  'C4': 759.7486588511862,
  'C5': 7395.251833070876,
  'C6': 6550.464908215372},
 {'C1': 807.1257565471984,
  'C2': 1881.551576131434,
  'C3': 30.59391498258557,
  'C4': 1940.0549530491269,
  'C5': 1093.0314346605865,
  'C6': 1020.8803106380971},
 {'C1': 56.380877607708975,
  'C2': 5508.389847519281,
  'C3': 214.77052706983454,
  'C4': 582.3858565407576,
  'C5': 794.5778814423682,
  'C6': 18116.790862777052},
 {'C1': 28

In [3]:
components_cfg, couplings_cfg, cities_cfg = load_configs()
DATES = default_dates()
manifest = []
for k, schedule in enumerate(tau_schedules):
    output_path = POLICY_DIR / f'policy_{k:03d}.parquet'
    if output_path.exists():
        print(f'[{k}] cached:', output_path)
        manifest.append({'k': k, 'tau': schedule, 'path': output_path.as_posix()})
        continue
    print(f'[{k}] simulating', schedule)
    df = run_with_tau(
        schedule,
        printer_ids=TRAIN_PRINTERS,
        dates=DATES,
        components_cfg=components_cfg,
        couplings_cfg=couplings_cfg,
        cities_cfg=cities_cfg,
    )
    table = table_from_dataframe(df, include_rul=False)
    pq.write_table(table, output_path, compression='snappy', version='2.6')
    add_rul_labels(output_path)
    manifest.append({'k': k, 'tau': schedule, 'path': output_path.as_posix()})
    print(f'[{k}] wrote {output_path} ({output_path.stat().st_size / 1e6:.1f} MB)')

[0] simulating {'C1': 155.59583785006157, 'C2': 1765.2107916409634, 'C3': 86.472130536287, 'C4': 1039.6324922070855, 'C5': 3882.5691163902784, 'C6': 13990.349494924487}


[0] wrote C:\Users\Sergi\OneDrive\Desktop\Industrial Engineering\Q8\TFG\Coding\hackupc2026\data\policy_runs\policy_000.parquet (39.1 MB)
[1] simulating {'C1': 611.1034528608234, 'C2': 537.1586666615513, 'C3': 154.47160942624942, 'C4': 943.5781632106193, 'C5': 636.9234158299065, 'C6': 1931.776617446712}


[1] wrote C:\Users\Sergi\OneDrive\Desktop\Industrial Engineering\Q8\TFG\Coding\hackupc2026\data\policy_runs\policy_001.parquet (39.1 MB)
[2] simulating {'C1': 1132.435398876648, 'C2': 3804.364631233711, 'C3': 378.572339510078, 'C4': 1712.3245117575705, 'C5': 1401.0340681294374, 'C6': 5507.817068918813}


[2] wrote C:\Users\Sergi\OneDrive\Desktop\Industrial Engineering\Q8\TFG\Coding\hackupc2026\data\policy_runs\policy_002.parquet (39.2 MB)
[3] simulating {'C1': 1381.2400315847656, 'C2': 2544.203495476921, 'C3': 42.96238310854736, 'C4': 759.7486588511862, 'C5': 7395.251833070876, 'C6': 6550.464908215372}


[3] wrote C:\Users\Sergi\OneDrive\Desktop\Industrial Engineering\Q8\TFG\Coding\hackupc2026\data\policy_runs\policy_003.parquet (39.1 MB)
[4] simulating {'C1': 807.1257565471984, 'C2': 1881.551576131434, 'C3': 30.59391498258557, 'C4': 1940.0549530491269, 'C5': 1093.0314346605865, 'C6': 1020.8803106380971}


...

[15] wrote C:\Users\Sergi\OneDrive\Desktop\Industrial Engineering\Q8\TFG\Coding\hackupc2026\data\policy_runs\policy_015.parquet (38.7 MB)
[16] simulating {'C1': 1753.243495691169, 'C2': 2661.1112517539764, 'C3': 102.80330499336696, 'C4': 634.7783865743859, 'C5': 4675.582580416165, 'C6': 2787.4309397747847}


[16] wrote C:\Users\Sergi\OneDrive\Desktop\Industrial Engineering\Q8\TFG\Coding\hackupc2026\data\policy_runs\policy_016.parquet (39.1 MB)
[17] simulating {'C1': 329.1740382529947, 'C2': 12672.658225572255, 'C3': 54.87851188487632, 'C4': 1111.0959430988062, 'C5': 3110.9801768629195, 'C6': 2407.6671568423612}


[17] wrote C:\Users\Sergi\OneDrive\Desktop\Industrial Engineering\Q8\TFG\Coding\hackupc2026\data\policy_runs\policy_017.parquet (39.1 MB)
[18] simulating {'C1': 202.56761280855213, 'C2': 3357.152540217728, 'C3': 131.2028559582582, 'C4': 177.72596280124557, 'C5': 4169.137482544622, 'C6': 3562.107526502361}


[18] wrote C:\Users\Sergi\OneDrive\Desktop\Industrial Engineering\Q8\TFG\Coding\hackupc2026\data\policy_runs\policy_018.parquet (39.1 MB)
[19] simulating {'C1': 123.95280210684123, 'C2': 10449.870872819978, 'C3': 35.353463820408386, 'C4': 197.87193103756428, 'C5': 1271.4636332081475, 'C6': 4918.231439959994}


[19] wrote C:\Users\Sergi\OneDrive\Desktop\Industrial Engineering\Q8\TFG\Coding\hackupc2026\data\policy_runs\policy_019.parquet (39.1 MB)


In [4]:
with MANIFEST_PATH.open('w', encoding='utf-8') as handle:
    json.dump({'tau_ranges': TAU_RANGES, 'seed': SEED, 'runs': manifest}, handle, indent=2)
print('Wrote manifest:', MANIFEST_PATH)
print(f'{len(manifest)} policy runs covering printer_ids {TRAIN_PRINTERS[0]}..{TRAIN_PRINTERS[-1]}')

Wrote manifest: C:\Users\Sergi\OneDrive\Desktop\Industrial Engineering\Q8\TFG\Coding\hackupc2026\data\policy_runs\manifest.json
20 policy runs covering printer_ids 0..69
